# MCP + LangGraph 学习版（Fireworks GLM-5）

这个 Notebook 版本专门为**学习**而设计：
- 把代码拆成多个可单独运行的 cell
- 每个 cell 提供清晰注释与说明
- 保留核心能力：MCP 工具、超时、重试、fallback

> 建议按顺序从上到下执行。

## Cell 1: 安装依赖（按需执行）

如果你当前环境缺少依赖，可以取消注释后执行。

In [ ]:
# %pip install -U langchain-openai langgraph langchain-mcp-adapters

## Cell 2: 导入模块

这里导入：`asyncio`、`dataclass`、`ChatOpenAI`、`MultiServerMCPClient`、`create_react_agent`。

In [ ]:
from __future__ import annotations

import asyncio
import os
from dataclasses import dataclass
from typing import Any

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

## Cell 3: 配置区（环境变量）

关键变量说明：
- `FIREWORKS_API_KEY`：Fireworks 鉴权 Key（必需）
- `FIREWORKS_BASE_URL`：OpenAI 兼容接口地址
- `FIREWORKS_PRIMARY_MODEL` / `FIREWORKS_FALLBACK_MODEL`：主/备模型
- `AGENT_TIMEOUT_SECONDS`：单次调用超时
- `AGENT_MAX_RETRIES`：失败重试次数

In [ ]:
@dataclass(frozen=True)
class Settings:
    # Fireworks OpenAI-compatible endpoint
    fireworks_base_url: str = os.getenv("FIREWORKS_BASE_URL", "https://api.fireworks.ai/inference/v1")

    # 主模型与兜底模型（默认都用 GLM-5）
    fireworks_primary_model: str = os.getenv("FIREWORKS_PRIMARY_MODEL", "accounts/fireworks/models/glm-5")
    fireworks_fallback_model: str = os.getenv("FIREWORKS_FALLBACK_MODEL", "accounts/fireworks/models/glm-5")

    # 会话与调用控制
    thread_id: str = os.getenv("AGENT_THREAD_ID", "conversation_id")
    request_timeout_s: float = float(os.getenv("AGENT_TIMEOUT_SECONDS", "25"))
    max_retries: int = int(os.getenv("AGENT_MAX_RETRIES", "2"))

## Cell 4: 核心执行器 AgentRunner

职责：
1. 连接 MCP Servers（Context7 + Met Museum）
2. 构建 primary/fallback 两个 agent
3. 提供带超时 + 重试的统一调用方法

In [ ]:
class AgentRunner:
    """封装 MCP 初始化、Agent 构建与稳健调用。"""

    def __init__(self, settings: Settings) -> None:
        self.settings = settings
        self.config = {"configurable": {"thread_id": settings.thread_id}}

        self.client = MultiServerMCPClient(
            {
                "context7": {
                    "url": "https://mcp.context7.com/mcp",
                    "transport": "streamable_http",
                },
                "met-museum": {
                    "command": "npx",
                    "args": ["-y", "metmuseum-mcp"],
                    "transport": "stdio",
                },
            }
        )

    async def build_agents(self) -> tuple[Any, Any]:
        """构建主/备两个 Fireworks GLM-5 Agent。"""
        tools = await self.client.get_tools()
        checkpointer = InMemorySaver()

        primary_model = ChatOpenAI(
            model=self.settings.fireworks_primary_model,
            base_url=self.settings.fireworks_base_url,
            api_key=os.getenv("FIREWORKS_API_KEY"),
        )
        fallback_model = ChatOpenAI(
            model=self.settings.fireworks_fallback_model,
            base_url=self.settings.fireworks_base_url,
            api_key=os.getenv("FIREWORKS_API_KEY"),
        )

        primary_agent = create_react_agent(
            model=primary_model,
            tools=tools,
            checkpointer=checkpointer,
        )
        fallback_agent = create_react_agent(
            model=fallback_model,
            tools=tools,
            checkpointer=checkpointer,
        )
        return primary_agent, fallback_agent

    async def invoke_with_retry(self, agent: Any, user_text: str, *, system_prompt: str | None = None) -> str:
        """带超时和指数退避重试的调用。"""
        messages: list[dict[str, str]] = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_text})

        last_error: Exception | None = None
        for attempt in range(self.settings.max_retries + 1):
            try:
                response = await asyncio.wait_for(
                    agent.ainvoke({"messages": messages}, config=self.config),
                    timeout=self.settings.request_timeout_s,
                )
                return response["messages"][-1].content
            except Exception as exc:  # noqa: BLE001
                last_error = exc
                if attempt >= self.settings.max_retries:
                    break
                await asyncio.sleep(0.8 * (2**attempt))

        raise RuntimeError(f"Agent call failed after retries: {last_error}") from last_error

## Cell 5: 初始化对象

执行后会创建：`settings`、`runner`、`primary_agent`、`fallback_agent`。

In [ ]:
settings = Settings()
runner = AgentRunner(settings)

primary_agent, fallback_agent = await runner.build_agents()
print("✅ Agents are ready")

## Cell 6: 启动测试（Boot Test）

先发一个固定问题，确认 Agent + MCP 工具链是否正常。

In [ ]:
system_prompt = (
    "You are a smart, useful agent with tools to access code library "
    "documentation and the Met Museum collection."
)

intro = await runner.invoke_with_retry(
    primary_agent,
    "Give a brief introduction of what you do and the tools you can access.",
    system_prompt=system_prompt,
)
print(intro)

## Cell 7: 单次提问函数（带 fallback）

这个函数便于你重复练习：优先主模型，失败自动切到 fallback。

In [ ]:
async def ask_once(query: str) -> str:
    try:
        return await runner.invoke_with_retry(primary_agent, query)
    except Exception as primary_error:  # noqa: BLE001
        print(f"⚠️ Primary model failed, fallback starts: {primary_error}")
        return await runner.invoke_with_retry(fallback_agent, query)

## Cell 8: 进行一次实际提问

把 `query` 改成你自己的问题。

In [ ]:
query = "请帮我介绍 MCP 是什么，以及这个示例里用到了哪些工具能力？"
answer = await ask_once(query)
print(answer)

## Cell 9: （可选）简易循环提问

> 在 Notebook 中无限循环交互体验一般，这里仅作学习示例。

In [ ]:
# while True:
#     q = input("你想问什么？(输入 quit 退出)\n> " ).strip()
#     if q.lower() in {"quit", "exit"}:
#         print("Bye")
#         break
#     if not q:
#         print("请输入有效问题")
#         continue
#     print(await ask_once(q))